# LIBRARY

In [1]:
# UTILS
import numpy as np
from collections import defaultdict
import os
import pickle
import copy
import os
import numpy as np
import pandas as pd
import json
from collections import defaultdict
from sklearn.model_selection import train_test_split
import datetime
import time
import argparse
import pickle
import re
import random
import sys

# MODEL
import datetime
import math
import numpy as np
import torch
from torch import nn
from torch.nn import Module, Parameter
import torch.nn.functional as F
from collections import defaultdict

# MAIN
import argparse
import os
import sys
import time
import datetime
import torch

# UTILS

In [ ]:
def data_input(opt):
    BASE_DIR = '/kaggle/input/PREPOCESSING/hasil_prepocessing'
    # BASE_DIR = './PREPOCESSING/hasil_prepocessing' 
    dataset_name='music4all'
    freq = 10
    topu = 8000
    PATH = 'music4all'
    n_music = 56047

    test_user_item_record = os.path.join(BASE_DIR, PATH,
                                         'baseline_te_new_freq{}_partition8.lst'.format(freq))
    train_user_item_record = os.path.join(BASE_DIR, PATH,
                                          'baseline_tr_freq{}_partition8.lst'.format(freq))

    music2artist_dic_record = os.path.join(BASE_DIR, PATH,
                                           '{}_music_index2artist_freq{}_topu{}'.format(dataset_name, freq, topu))
    music2album_dic_record = os.path.join(BASE_DIR, PATH,
                                          '{}_music_index2album_freq{}_topu{}'.format(dataset_name, freq, topu))
    music2artist_dic = pickle.load(open(music2artist_dic_record, 'rb'))
    music2album_dic = pickle.load(open(music2album_dic_record, 'rb'))

    test_user_item_file = open(test_user_item_record, 'rb')
    train_user_item_file = open(train_user_item_record, 'rb')

    test_lines = test_user_item_file.readlines()
    train_lines = train_user_item_file.readlines()

    # List untuk menyimpan data user dan interaksi
    train_user, train_music, train_artist, train_album = [], [], [], []
    test_user, test_music, test_artist, test_album = [], [], [], []

    # Dictionary untuk menyimpan lagu yang pernah didengar user (untuk filtering)
    usr2music_record_dic = defaultdict(set)

    flag = 1
    train_lenth,test_lenth=0,0
    user_set=set()

    # Proses data training
    for line in train_lines:
        if flag:
            flag = 0
            continue
        line = line.decode()
        user_id = int(line.split(',')[0])
        user_set.add(user_id)

        items_id = line.split(',')[1].split(':')
        int_items_id = list(map(int, items_id))
        int_items_id_plus = np.array(int_items_id) + 1
        train_lenth += len(int_items_id)

        for item in int_items_id_plus:
            usr2music_record_dic[user_id].add(item)

        train_user.append(user_id)
        train_music.append(int_items_id_plus.tolist())

        train_artist.append([int(music2artist_dic[x]) + 1 for x in int_items_id])
        train_album.append([int(music2album_dic[x]) + 1 for x in int_items_id])

    # Proses data test
    for line in test_lines:
        line = line.decode()
        user_id = int(line.split(',')[0])
        user_set.add(user_id)

        items_id = line.split(',')[1].split(':')
        int_items_id = list(map(int, items_id))
        int_items_id_plus = np.array(int_items_id) + 1
        test_lenth += len(int_items_id)

        test_user.append(user_id)
        test_music.append(int_items_id_plus.tolist())

        test_artist.append([int(music2artist_dic[x]) + 1 for x in int_items_id])
        test_album.append([int(music2album_dic[x]) + 1 for x in int_items_id])

    test_user_item_file.close()
    train_user_item_file.close()

    # Gabungkan data user dan interaksinya
    train_data = train_user, train_music, train_artist, train_album
    test_data = test_user, test_music, test_artist, test_album

    # Bagi data train menjadi window sequence
    train_data = data_split(train_data, usr2music_record_dic, opt, '', is_test=False)

    test_new_data = copy.deepcopy(test_data)
    test_new_data = data_split(test_new_data, usr2music_record_dic, opt, 'next-new-item', is_test=True)
    test_data = data_split(test_data, usr2music_record_dic, opt, 'next-one-item', is_test=True)

    return train_data, test_data, test_new_data

In [3]:
def data_split(data, usr2music_record_dic, opt, mode, is_test=False):
    windowLenth = opt.windowLenth
    data_size = opt.data_size
    step = opt.slide_step  

    user, target, music_seq, artist_seq, album_seq = [], [], [], [], []

    if is_test == False:
        # Bagi data train menjadi window sequence
        for user_id, music_slice, artist_slice, album_slice in zip(data[0], data[1], data[2], data[3]):
            for i in range(windowLenth, int(len(music_slice) * data_size), step):
                user.append(user_id)
                target += [music_slice[i]]
                music_seq.append(music_slice[i - windowLenth:i])
                artist_seq.append(artist_slice[i - windowLenth:i])
                album_seq.append(album_slice[i - windowLenth:i])
    else:
        # Data test dibagi berdasarkan mode tertentu
        if mode == 'next-new-item':
            # Target belum pernah muncul di window maupun riwayat user
            for user_id, music_slice, artist_slice, album_slice in zip(data[0], data[1], data[2], data[3]):
                for i in range(windowLenth, int(len(music_slice))):
                    t = music_slice[i]
                    if t in usr2music_record_dic[user_id] or t in music_slice[i - windowLenth:i]:
                        continue
                    user.append(user_id)
                    target += [music_slice[i]]
                    music_seq.append(music_slice[i - windowLenth:i])
                    artist_seq.append(artist_slice[i - windowLenth:i])
                    album_seq.append(album_slice[i - windowLenth:i])

        elif mode == 'next-one-item':
            # Gunakan item berikutnya secara langsung
            for user_id, music_slice, artist_slice, album_slice in zip(data[0], data[1], data[2], data[3]):
                for i in range(windowLenth, int(len(music_slice))):
                    user.append(user_id)
                    target += [music_slice[i]]
                    music_seq.append(music_slice[i - windowLenth:i])
                    artist_seq.append(artist_slice[i - windowLenth:i])
                    album_seq.append(album_slice[i - windowLenth:i])

    return user, target, music_seq, artist_seq, album_seq

In [4]:
def build_graph(music_seq, artist_seq, album_seq):
    matrix=[]
    music_alias_input, artist_alias_input, album_alias_input = [], [], []

    # Inisialisasi item graph
    items, len_list = [], []

    for music_slice, artist_slice, album_slice in zip(music_seq, artist_seq, album_seq):
        a = np.unique(music_slice + artist_slice + album_slice)
        items.append(a)
        len_list.append(a.shape[0] + 1)  # Tambah 1 sebagai placeholder untuk node user

    max_node = max(len_list)  # Panjang maksimum node dalam satu graph

    # Padding semua item agar memiliki panjang sama
    for i in range(len(items)):
        items[i] = np.pad(items[i], (0, max_node - len_list[i]), 'constant', constant_values=(0))

    for i in range(len(items)):
        # Konversi item sequence menjadi indeks dalam items (alias untuk hidden layer)
        music_alias_input.append([np.where(items[i] == m)[0][0] + 1 for m in music_seq[i]])
        artist_alias_input.append([np.where(items[i] == m)[0][0] + 1 for m in artist_seq[i]])
        album_alias_input.append([np.where(items[i] == m)[0][0] + 1 for m in album_seq[i]])

        # Buat adjacency matrix (graph) untuk sequence
        u_A = np.zeros((max_node, max_node))  # Graph dimulai dari index 0 (user)

        # Tambahkan edge dari user ke item pertama (lagu, artis, album)
        u_A[0][np.where(items[i] == music_seq[i][0])[0][0] + 1] = 1
        u_A[0][np.where(items[i] == artist_seq[i][0])[0][0] + 1] = 1
        u_A[0][np.where(items[i] == album_seq[i][0])[0][0] + 1] = 1

        pre = -1  # Menyimpan lagu sebelumnya untuk membuat edge berurutan
        for music, artist, album in zip(music_seq[i], artist_seq[i], album_seq[i]):
            if pre != -1:
                pre_music_pos = np.where(items[i] == pre)[0][0] + 1
            now_music_pos = np.where(items[i] == music)[0][0] + 1
            artist_pos = np.where(items[i] == artist)[0][0] + 1
            album_pos = np.where(items[i] == album)[0][0] + 1

            # Buat edge dari lagu sebelumnya ke lagu sekarang
            if pre != -1:
                u_A[pre_music_pos][now_music_pos] = 1
            pre = music

            # Tambahkan edge: artist → lagu, album → lagu
            if artist != 0:
                u_A[artist_pos][now_music_pos] = 1
            if album != 0:
                u_A[album_pos][now_music_pos] = 1

            # Tambahkan edge dua arah antara artist dan album jika keduanya ada
            if artist and album:
                u_A[album_pos][artist_pos] = 1
                u_A[artist_pos][album_pos] = 1

        # Normalisasi adjacency matrix ke dua arah: masuk dan keluar
        u_sum_in = np.sum(u_A, 0)
        u_sum_in[np.where(u_sum_in == 0)] = 1  # Hindari pembagian nol
        u_A_in = np.divide(u_A, u_sum_in)

        u_sum_out = np.sum(u_A, 1)
        u_sum_out[np.where(u_sum_out == 0)] = 1
        u_A_out = np.divide(u_A.transpose(), u_sum_out)

        # Gabungkan dua versi adjacency matrix
        u_A = np.concatenate([u_A_out, u_A_in]).transpose()
        matrix.append(u_A)

    return matrix, items, music_alias_input, artist_alias_input, album_alias_input

In [5]:
class Data():
    def __init__(self, data, opt, shuffle=False):
        usr = data[0]
        target = data[1]
        music_seq = data[2]
        artist_seq = data[3]
        album_seq = data[4]

        # Bangun struktur graf
        matrix, items, music_alias_input, artist_alias_input, album_alias_input = build_graph(music_seq, artist_seq, album_seq)

        self.usr = np.asarray(usr)
        self.target = np.asarray(target)
        self.items = np.asarray(items)
        self.music_alias_input = np.asarray(music_alias_input)
        self.artist_alias_input = np.asarray(artist_alias_input)
        self.album_alias_input = np.asarray(album_alias_input)
        self.matrix = np.asarray(matrix)

        self.windowLenth = opt.windowLenth
        self.shuffle = shuffle

    def generate_batch(self, batch_size):
        length = len(self.target)

        if self.shuffle:
            shuffled_arg = np.arange(length)
            np.random.shuffle(shuffled_arg)

            # Terapkan pengacakan ke seluruh input data
            self.usr = self.usr[shuffled_arg]
            self.target = self.target[shuffled_arg]
            self.items = self.items[shuffled_arg]
            self.matrix = self.matrix[shuffled_arg]

            self.music_alias_input = self.music_alias_input[shuffled_arg]
            self.artist_alias_input = self.artist_alias_input[shuffled_arg]
            self.album_alias_input = self.album_alias_input[shuffled_arg]

        n_batch = int(length / batch_size)
        if length % batch_size != 0:
            n_batch += 1

        slices = np.split(np.arange(n_batch * batch_size), n_batch)
        slices[-1] = slices[-1][:(length - batch_size * (n_batch - 1))]

        return slices

# MODEL

In [8]:
class GNN(Module):
    def __init__(self, hidden_size, step=1):
        super(GNN, self).__init__()

        # Jumlah langkah propagasi dalam graph neural network
        self.step = step

        # Ukuran vektor tersembunyi (hidden state)
        self.hidden_size = hidden_size

        # Ukuran input ke sel GNN adalah gabungan dari edge_in dan edge_out
        self.input_size = hidden_size * 2

        # Ukuran gate GRU (reset, update, dan new gate)
        self.gate_size = 3 * hidden_size

        # Parameter yang akan dipelajari untuk GNNCell
        self.w_ih = Parameter(torch.Tensor(self.gate_size, self.input_size))  # Input weight
        self.w_hh = Parameter(torch.Tensor(self.gate_size, self.hidden_size))  # Hidden state weight
        self.b_ih = Parameter(torch.Tensor(self.gate_size))  # Bias untuk input
        self.b_hh = Parameter(torch.Tensor(self.gate_size))  # Bias untuk hidden
        self.b_iah = Parameter(torch.Tensor(self.hidden_size))  # Bias untuk edge in
        self.b_oah = Parameter(torch.Tensor(self.hidden_size))  # Bias untuk edge out

        # Layer linear untuk fitur edge in, edge out, dan fusion edge
        self.linear_edge_in = nn.Linear(self.hidden_size, self.hidden_size, bias=True)
        self.linear_edge_out = nn.Linear(self.hidden_size, self.hidden_size, bias=True)
        self.linear_edge_f = nn.Linear(self.hidden_size, self.hidden_size, bias=True)

    def GNNCell(self, A, hidden):
        # Fungsi sel utama untuk menghitung update pada tiap node berdasarkan matriks adjacency dan hidden state saat ini

        # Hitung input dari sisi masuk dan keluar pada graph
        input_in = torch.matmul(A[:, :, :A.shape[1]], self.linear_edge_in(hidden)) + self.b_iah
        input_out = torch.matmul(A[:, :, A.shape[1]: 2 * A.shape[1]], self.linear_edge_out(hidden)) + self.b_oah

        # Gabungkan input masuk dan keluar
        inputs = torch.cat([input_in, input_out], 2)

        # Hitung gate input dan hidden (mirip GRU)
        gi = F.linear(inputs, self.w_ih, self.b_ih)
        gh = F.linear(hidden, self.w_hh, self.b_hh)

        # Bagi menjadi tiga bagian: reset gate, input gate, dan new gate
        i_r, i_i, i_n = gi.chunk(3, 2)
        h_r, h_i, h_n = gh.chunk(3, 2)

        # Hitung aktivasi gate
        resetgate = torch.sigmoid(i_r + h_r)
        inputgate = torch.sigmoid(i_i + h_i)
        newgate = torch.tanh(i_n + resetgate * h_n)

        # Update hidden state menggunakan formula GRU
        hy = newgate + inputgate * (hidden - newgate)

        return hy

    def forward(self, A, hidden):
        # Fungsi forward yang melakukan propagasi graph selama beberapa step
        for i in range(self.step):
            hidden = self.GNNCell(A, hidden)  # Update hidden state di tiap langkah
        return hidden  # Kembalikan hidden state terakhir setelah propagasi

In [ ]:
class SessionGraph(Module):
    def __init__(self, opt, n_usr, n_album, n_artist, n_music):
        super(SessionGraph, self).__init__()
        self.hidden_size = opt.hiddenSize
        self.n_usr = n_usr
        self.n_music = n_music
        self.n_item = n_album + n_artist + n_music + 1  
        self.batch_size = opt.batchSize
        self.item_embedding = nn.Embedding(self.n_item, self.hidden_size)
        self.usr_embedding = nn.Embedding(self.n_usr, self.hidden_size)

        self.gnn = GNN(self.hidden_size, step=opt.step)

        self.linear_one_1 = nn.Linear(self.hidden_size, self.hidden_size, bias=True)
        self.linear_two_1 = nn.Linear(self.hidden_size, self.hidden_size, bias=True)

        self.linear_one_2 = nn.Linear(self.hidden_size, self.hidden_size, bias=True)
        self.linear_two_2 = nn.Linear(self.hidden_size, self.hidden_size, bias=True)

        self.linear_one_3 = nn.Linear(self.hidden_size, self.hidden_size, bias=True)
        self.linear_two_3 = nn.Linear(self.hidden_size, self.hidden_size, bias=True)

        self.linear_layer2 = nn.Linear(self.hidden_size, self.hidden_size, bias=True)
        self.linear_transform = nn.Linear(self.hidden_size * 4, self.hidden_size, bias=True)
        self.linear_transform1 = nn.Linear(self.hidden_size * 3, self.hidden_size, bias=True)
        self.linear_transform2 = nn.Linear(self.hidden_size * 3, self.hidden_size, bias=True)
        self.linear_transform3 = nn.Linear(self.hidden_size * 2, self.hidden_size, bias=True)

        self.loss_function = nn.CrossEntropyLoss()
        self.optimizer = torch.optim.Adam(self.parameters(), lr=opt.lr, weight_decay=opt.l2)
        self.scheduler = torch.optim.lr_scheduler.StepLR(self.optimizer, step_size=opt.lr_dc_step, gamma=opt.lr_dc)
        self.reset_parameters()

    def reset_parameters(self):
        stdv = 1.0 / math.sqrt(self.hidden_size)
        for weight in self.parameters():
            weight.data.uniform_(-stdv, stdv)

    def compute_scores(self, usr_embedding, music_embedding, artist_embedding, album_embedding):  

        # alpha
        ht = usr_embedding.view(usr_embedding.shape[0], usr_embedding.shape[1], 1)  # batch_size x latent_size

        last_music_embedding = music_embedding[:, -1, :]  
        last_artist_embedding = artist_embedding[:, -1, :]
        last_album_embedding = album_embedding[:, -1, :]

        q1_1 = self.linear_one_1(last_music_embedding).view(last_music_embedding.shape[0], 1,
                                                            last_music_embedding.shape[1])
        # batch_size x seq_length x latent_size
        q1_2 = self.linear_two_1(music_embedding)

        q2_1 = self.linear_one_2(last_artist_embedding).view(last_artist_embedding.shape[0], 1,
                                                             last_artist_embedding.shape[1])
        q2_2 = self.linear_two_2(artist_embedding)

        q3_1 = self.linear_one_3(last_album_embedding).view(last_album_embedding.shape[0], 1,
                                                            last_album_embedding.shape[1])
        q3_2 = self.linear_two_3(album_embedding)

        alpha = torch.bmm((torch.tanh(q1_1 + q1_2)), ht)  # batch x node x 1
        beta = torch.bmm((torch.tanh(q2_1 + q2_2)), ht)
        delta = torch.bmm((torch.tanh(q3_1 + q3_2)), ht)

        alpha = torch.softmax(alpha, dim=1)
        beta = torch.softmax(beta, dim=1)
        delta = torch.softmax(delta, dim=1)
        a = torch.sum(alpha * music_embedding, 1)  # batch x hidden
        b = torch.sum(beta * artist_embedding, 1)
        c = torch.sum(delta * album_embedding, 1)

        layernorm = trans_to_cuda(nn.LayerNorm(a.shape[1], eps=1e-6))
        a = layernorm(a)
        b = layernorm(b)
        c = layernorm(c)

        st = self.linear_transform1(torch.cat([a, b, c], 1))  # short-term
        all = self.linear_transform3(torch.cat([usr_embedding, st], 1))

        item_embedding = self.item_embedding.weight[1:self.n_music + 1]  # n_nodes x latent_size
        scores = torch.matmul(all, item_embedding.transpose(1, 0))

        return scores  # batch_size x item_num

    def forward(self, item, usr, A):
        # usr batch x 1
        # item: batch x max_node
        h1 = self.item_embedding(item)  
        h2 = self.usr_embedding(usr.view(usr.shape[0], 1))
        hidden = torch.cat([h2, h1], dim=1)
        hidden = self.gnn(A, hidden) 
        return hidden

In [10]:
def trans_to_cuda(variable):
    if torch.cuda.is_available():
        return variable.cuda()
    else:
        return variable

In [11]:
def trans_to_cpu(variable):
    if torch.cuda.is_available():
        return variable.cpu()
    else:
        return variable

In [ ]:
def forward(model, i, data):  
    A = data.matrix[i]
    usr = data.usr[i]
    items = data.items[i]  

    music_alias_input = data.music_alias_input[i]
    artist_alias_input = data.artist_alias_input[i]
    album_alias_input = data.album_alias_input[i]

    items = trans_to_cuda(torch.Tensor(items).long())
    usr = trans_to_cuda(torch.Tensor(usr).long())
    A = trans_to_cuda(torch.Tensor(A).float())

    hidden = model(items, usr, A) 
    usr_embedding = hidden[:, 0, :]

    music_embedding = torch.stack([hidden[i][music_alias_input[i]] for i in torch.arange(len(music_alias_input)).long()])
    artist_embedding = torch.stack([hidden[i][artist_alias_input[i]] for i in torch.arange(len(artist_alias_input)).long()])
    album_embedding = torch.stack([hidden[i][album_alias_input[i]] for i in torch.arange(len(album_alias_input)).long()])

    targets = data.target[i]
    return targets, model.compute_scores(usr_embedding, music_embedding, artist_embedding, album_embedding)

In [13]:
def predict(model, test_data):
    N = 21
    hit_res = np.zeros(N)
    mrr_res = np.zeros(N)

    model.eval()
    hit = defaultdict(list)
    mrr = defaultdict(list)
    slices = test_data.generate_batch(model.batch_size)
    for i in slices:
        targets, scores = forward(model, i, test_data)
        sub_scores = scores.topk(N - 1)[1]
        sub_scores = trans_to_cpu(sub_scores).detach().numpy()

        for score, target in zip(sub_scores, targets):
            target = target - 1
            for topN in range(1, N):
                topN_item = score[:topN]
                hit[topN].append(np.isin(target, topN_item))
                common = []
                if target in topN_item:
                    common.append(target)
                    mrr[topN].append(1 / (np.where(topN_item == target)[0][0] + 1))
                else:
                    mrr[topN].append(0)

    for topN in range(1, N):
        hit_res[topN] = np.mean(hit[topN]) * 100
        mrr_res[topN] = np.mean(mrr[topN]) * 100

    print("hit:\n{}".format(hit_res[1:]))
    print("mrr:\n{}".format(mrr_res[1:]))

    return hit_res[1:], mrr_res[1:]

In [14]:
def train_test(model, train_data, test_data, test_new_data):
    print('start training: ')
    model.scheduler.step()
    model.train()
    total_loss = 0.0
    slices = train_data.generate_batch(model.batch_size)
    for i, j in zip(slices, np.arange(len(slices))):
        model.optimizer.zero_grad()
        targets, scores = forward(model, i, train_data)
        targets = trans_to_cuda(torch.Tensor(targets).long())
        loss = model.loss_function(scores, targets-1)
        loss.backward()
        model.optimizer.step()
        total_loss += loss
        if j % int(len(slices) / 5 + 1) == 0:
            print('[%d/%d] Loss: %.4f' % (j, len(slices), loss.item()))
    print('\tLoss:\t%.3f' % total_loss)

    print('start predicting (next-one): ')
    hit_next_one, mrr_next_one = predict(model, test_data)

    print('start predicting (next-new): ')
    hit_next_new, mrr_next_new = predict(model, test_new_data)

    return (hit_next_one, mrr_next_one), (hit_next_new, mrr_next_new)

# MAIN


In [14]:
SEED = 42
sys.setrecursionlimit(100000)
torch.manual_seed(1000)
torch.backends.cudnn.deterministic = True

parser = argparse.ArgumentParser()

parser.add_argument('--dataset', default='music4all', help='dataset name: music4all')
parser.add_argument('--detail', default='ModelTA_WithoutDynamic', help='Deskripsi kode untuk pembeda')
parser.add_argument('--data_size', default=1, type=float, help='Bagian data yang digunakan')
parser.add_argument('--slide_step', default=1, type=int, help='Langkah sliding window')
parser.add_argument('--windowLenth', type=int, default=3, help='Panjang maksimum sliding window')
parser.add_argument('--batchSize', type=int, default=256, help='Ukuran batch input')
parser.add_argument('--hiddenSize', type=int, default=100, help='Ukuran hidden state')
parser.add_argument('--epoch', type=int, default=10, help='Jumlah epoch untuk training')
parser.add_argument('--lr', type=float, default=0.001, help='Learning rate')
parser.add_argument('--lr_dc', type=float, default=0.1, help='Tingkat penurunan learning rate')
parser.add_argument('--lr_dc_step', type=int, default=3, help='Jumlah langkah sebelum learning rate turun')
parser.add_argument('--l2', type=float, default=1e-5, help='L2 penalty')
parser.add_argument('--step', type=int, default=1, help='Jumlah propagasi GNN')

opt = parser.parse_args(args=[])

# Checkpoint configuration untuk Kaggle
CHECKPOINT_INPUT_DIR = '/kaggle/input/checkpoint/pytorch/default/1'
CHECKPOINT_OUTPUT_DIR = '/kaggle/working/'
SAVE_EVERY = 5  # Save checkpoint every 5 epochs

start_time = time.time()

def save_checkpoint(model, epoch, results_next_one, results_next_new):
    """Simpan checkpoint model ke /kaggle/working"""
    os.makedirs(CHECKPOINT_OUTPUT_DIR, exist_ok=True)
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': model.optimizer.state_dict(),
        'scheduler_state_dict': model.scheduler.state_dict(),
        'results_next_one': results_next_one,
        'results_next_new': results_next_new,
        'opt': opt
    }
    checkpoint_path = os.path.join(CHECKPOINT_OUTPUT_DIR, f'model_withoutdynamic_checkpoint_epoch_{epoch + 1}.pt')
    torch.save(checkpoint, checkpoint_path)
    print(f"Checkpoint saved: {checkpoint_path}")
    return checkpoint_path

def find_checkpoint_in_input():
    """Cari checkpoint di /kaggle/input/checkpoint/pytorch/default/1"""
    checkpoint_files = []
    
    if os.path.exists(CHECKPOINT_INPUT_DIR):
        print(f"Scanning checkpoint input directory: {CHECKPOINT_INPUT_DIR}")
        files = os.listdir(CHECKPOINT_INPUT_DIR)
        print(f"Files found: {files}")
        
        for file in files:
            if file.startswith('model_withoutdynamic_checkpoint_epoch_') and file.endswith('.pt'):
                full_path = os.path.join(CHECKPOINT_INPUT_DIR, file)
                try:
                    epoch_num = int(file.split('_epoch_')[1].split('.pt')[0])
                    checkpoint_files.append((epoch_num, full_path))
                    print(f"Found checkpoint: {file} (epoch {epoch_num})")
                except:
                    continue
    else:
        print(f"Checkpoint input directory not found: {CHECKPOINT_INPUT_DIR}")
    
    return checkpoint_files

def find_checkpoint_in_working():
    """Cari checkpoint di /kaggle/working dari session sebelumnya"""
    checkpoint_files = []
    
    if os.path.exists(CHECKPOINT_OUTPUT_DIR):
        files = os.listdir(CHECKPOINT_OUTPUT_DIR)
        for file in files:
            if file.startswith('model_withoutdynamic_checkpoint_epoch_') and file.endswith('.pt'):
                full_path = os.path.join(CHECKPOINT_OUTPUT_DIR, file)
                try:
                    epoch_num = int(file.split('_epoch_')[1].split('.pt')[0])
                    checkpoint_files.append((epoch_num, full_path))
                except:
                    continue
    
    return checkpoint_files

def load_latest_checkpoint(model):
    """Load checkpoint terbaru dari input atau working directory"""
    print("Checking for existing checkpoints...")
    
    # Cari checkpoint di kedua lokasi
    input_checkpoints = find_checkpoint_in_input()
    working_checkpoints = find_checkpoint_in_working()
    
    # Gabungkan dan cari yang terbaru
    all_checkpoints = input_checkpoints + working_checkpoints
    
    if not all_checkpoints:
        print("No checkpoint found. Starting from scratch.")
        return 0, [], []
    
    # Ambil checkpoint dengan epoch tertinggi
    latest_epoch, latest_path = max(all_checkpoints, key=lambda x: x[0])
    
    print(f"Found latest checkpoint: {latest_path} (epoch {latest_epoch})")
    try:
        from argparse import Namespace
        import torch.serialization
        torch.serialization.add_safe_globals([Namespace])
        checkpoint = torch.load(latest_path, map_location='cpu')
        model.load_state_dict(checkpoint['model_state_dict'])
        model.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        model.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        results_next_one = checkpoint.get('results_next_one', [])
        results_next_new = checkpoint.get('results_next_new', [])
        print(f"Resuming training from epoch {start_epoch}")
        return start_epoch, results_next_one, results_next_new
    except Exception as e:
        print(f"Error loading checkpoint: {e}")
        print("Starting training from scratch")
        return 0, [], []

def main():
    print('Model Tanpa Preferensi Dinamis')
    print(f"Checkpoint input directory: {CHECKPOINT_INPUT_DIR}")
    print(f"Checkpoint output directory: {CHECKPOINT_OUTPUT_DIR}")

    opt.dataset = 'music4all'
    print(f'============= dataset: {opt.dataset} =============')

    n_music = 56047
    n_artist = 9348
    n_album = 21478
    n_usr = 8000

    print("--------------- Data Input ---------------")
    train_data_raw, test_data_raw, test_new_data_raw = data_input(opt)

    print(f"Total split train records: {len(train_data_raw[0])}")
    print(f"Total split test records (next-one): {len(test_data_raw[0])}")
    print(f"Total split test records (next-new): {len(test_new_data_raw[0])}")

    train_data = Data(train_data_raw, opt)
    test_data = Data(test_data_raw, opt)
    test_new_data = Data(test_new_data_raw, opt)

    model = trans_to_cuda(SessionGraph(opt, n_usr, n_album, n_artist, n_music))

    # Auto-load checkpoint jika ada
    start_epoch, results_next_one, results_next_new = load_latest_checkpoint(model)
    
    if start_epoch >= opt.epoch:
        print(f"Training already completed (epoch {start_epoch} >= {opt.epoch})")
        print("To continue training, increase --epoch parameter")
        return

    best_hit_next_one = None
    best_mrr_next_one = None
    best_hit_next_new = None
    best_mrr_next_new = None

    # Training loop dengan checkpoint otomatis
    for epoch in range(start_epoch, opt.epoch):
        print('-------------------------------------------------------')
        print(f'epoch: {epoch}')
        
        try:
            (hit_next_one, mrr_next_one), (hit_next_new, mrr_next_new) = train_test(model, train_data, test_data, test_new_data)
            sys.stdout.flush()

            best_hit_next_one = hit_next_one
            best_mrr_next_one = mrr_next_one
            best_hit_next_new = hit_next_new
            best_mrr_next_new = mrr_next_new

            result_one = {
                'epoch': epoch + 1,
                'hitrate10': float(best_hit_next_one[9]),
                'hitrate20': float(best_hit_next_one[19]),
                'mrr10': float(best_mrr_next_one[9]),
                'mrr20': float(best_mrr_next_one[19]),
            }
            result_new = {
                'epoch': epoch + 1,
                'hitrate10': float(best_hit_next_new[9]),
                'hitrate20': float(best_hit_next_new[19]),
                'mrr10': float(best_mrr_next_new[9]),
                'mrr20': float(best_mrr_next_new[19]),
            }

            results_next_one.append(result_one)
            results_next_new.append(result_new)

            # Auto-save checkpoint setiap N epoch
            if (epoch + 1) % SAVE_EVERY == 0:
                save_checkpoint(model, epoch, results_next_one, results_next_new)
                print(f"Auto-saved checkpoint at epoch {epoch + 1}")

        except Exception as e:
            print(f"Error during training at epoch {epoch}: {e}")
            # Simpan emergency checkpoint
            emergency_path = save_checkpoint(model, epoch - 1, results_next_one, results_next_new)
            print(f"Emergency checkpoint saved: {emergency_path}")
            raise e

    # Simpan final checkpoint jika training selesai
    if opt.epoch > 0:
        final_path = save_checkpoint(model, opt.epoch - 1, results_next_one, results_next_new)
        print(f"Final checkpoint saved: {final_path}")

    del model

    end_time = time.time()
    duration_hours = (end_time - start_time) / 3600
    print(f"\nTotal waktu yang dihabiskan: {duration_hours:.2f} jam")

    results_one_df = pd.DataFrame(results_next_one)
    results_new_df = pd.DataFrame(results_next_new)

    # Simpan hasil ke /kaggle/working dengan timestamp
    results_one_path = os.path.join(CHECKPOINT_OUTPUT_DIR, f'results_model_withoutdynamic_next_one.csv')
    results_new_path = os.path.join(CHECKPOINT_OUTPUT_DIR, f'results_model_withoutdynamic_next_new.csv')

    results_one_df.to_csv(results_one_path, index=False)
    results_new_df.to_csv(results_new_path, index=False)

    print(f"Saved results to:\n  {results_one_path}\n  {results_new_path}")

if __name__ == '__main__':
    main()

Model Tanpa Preferensi Dinamis
Checkpoint input directory: /kaggle/input/checkpoint/pytorch/default/1
Checkpoint output directory: /kaggle/working/
============= dataset: music4all =============
--------------- Data Input ---------------
Total split train records: 2471719
Total split test records (next-one): 603960
Total split test records (next-new): 272431
Checking for existing checkpoints...
Checkpoint input directory not found: /kaggle/input/checkpoint/pytorch/default/1
No checkpoint found. Starting from scratch.
-------------------------------------------------------
epoch: 0
start training: 


/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:227: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


[0/9656] Loss: 10.9936
[1932/9656] Loss: 8.0713
[3864/9656] Loss: 8.4758
[5796/9656] Loss: 3.9764
[7728/9656] Loss: 9.5195
	Loss:	68250.383
start predicting (next-one): 
hit:
[12.09186039 18.10484138 21.90294059 24.65693092 26.82561759 28.71299424
 30.24389032 31.55457315 32.66855421 33.60586794 34.44714882 35.18875422
 35.86661368 36.47178621 37.01039804 37.5076164  37.99258229 38.45718259
 38.880555   39.27495198]
mrr:
[12.09186039 15.09835088 16.36438395 17.05288154 17.48661887 17.80118165
 18.01988109 18.18371644 18.30749211 18.40122349 18.47770357 18.53950402
 18.59164705 18.63487366 18.67078112 18.70185726 18.73038467 18.7561958
 18.77847856 18.7981984 ]
start predicting (next-new): 
hit:
[ 6.20303857 10.62140505 13.60968465 15.87044059 17.69842639 19.23496225
 20.51198285 21.59482585 22.56314443 23.38206739 24.09747789 24.76186631
 25.37156197 25.92142598 26.40411701 26.86992303 27.3192111  27.7417034
 28.14877896 28.51914797]
mrr:
[ 6.20303857  8.41222181  9.40831501  9.9735039

In [15]:
SEED = 42
sys.setrecursionlimit(100000)
torch.manual_seed(1000)
torch.backends.cudnn.deterministic = True

parser = argparse.ArgumentParser()

parser.add_argument('--dataset', default='music4all', help='dataset name: music4all')
parser.add_argument('--detail', default='ModelTA_WithoutDynamic', help='Deskripsi kode untuk pembeda')
parser.add_argument('--data_size', default=1, type=float, help='Bagian data yang digunakan')
parser.add_argument('--slide_step', default=1, type=int, help='Langkah sliding window')
parser.add_argument('--windowLenth', type=int, default=3, help='Panjang maksimum sliding window')
parser.add_argument('--batchSize', type=int, default=256, help='Ukuran batch input')
parser.add_argument('--hiddenSize', type=int, default=100, help='Ukuran hidden state')
parser.add_argument('--epoch', type=int, default=20, help='Jumlah epoch untuk training')
parser.add_argument('--lr', type=float, default=0.001, help='Learning rate')
parser.add_argument('--lr_dc', type=float, default=0.1, help='Tingkat penurunan learning rate')
parser.add_argument('--lr_dc_step', type=int, default=3, help='Jumlah langkah sebelum learning rate turun')
parser.add_argument('--l2', type=float, default=1e-5, help='L2 penalty')
parser.add_argument('--step', type=int, default=1, help='Jumlah propagasi GNN')

opt = parser.parse_args(args=[])

# Checkpoint configuration untuk Kaggle
CHECKPOINT_INPUT_DIR = '/kaggle/input/checkpoint_model_withoutdynamic_epoch10/pytorch/default/1'
CHECKPOINT_OUTPUT_DIR = '/kaggle/working/'
SAVE_EVERY = 5  # Save checkpoint every 5 epochs

start_time = time.time()

def save_checkpoint(model, epoch, results_next_one, results_next_new):
    """Simpan checkpoint model ke /kaggle/working"""
    os.makedirs(CHECKPOINT_OUTPUT_DIR, exist_ok=True)
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': model.optimizer.state_dict(),
        'scheduler_state_dict': model.scheduler.state_dict(),
        'results_next_one': results_next_one,
        'results_next_new': results_next_new,
        'opt': opt
    }
    checkpoint_path = os.path.join(CHECKPOINT_OUTPUT_DIR, f'model_withoutdynamic_checkpoint_epoch_{epoch + 1}.pt')
    torch.save(checkpoint, checkpoint_path)
    print(f"Checkpoint saved: {checkpoint_path}")
    return checkpoint_path

def find_checkpoint_in_input():
    """Cari checkpoint di /kaggle/input/checkpoint/pytorch/default/1"""
    checkpoint_files = []
    
    if os.path.exists(CHECKPOINT_INPUT_DIR):
        print(f"Scanning checkpoint input directory: {CHECKPOINT_INPUT_DIR}")
        files = os.listdir(CHECKPOINT_INPUT_DIR)
        print(f"Files found: {files}")
        
        for file in files:
            if file.startswith('model_withoutdynamic_checkpoint_epoch_') and file.endswith('.pt'):
                full_path = os.path.join(CHECKPOINT_INPUT_DIR, file)
                try:
                    epoch_num = int(file.split('_epoch_')[1].split('.pt')[0])
                    checkpoint_files.append((epoch_num, full_path))
                    print(f"Found checkpoint: {file} (epoch {epoch_num})")
                except:
                    continue
    else:
        print(f"Checkpoint input directory not found: {CHECKPOINT_INPUT_DIR}")
    
    return checkpoint_files

def find_checkpoint_in_working():
    """Cari checkpoint di /kaggle/working dari session sebelumnya"""
    checkpoint_files = []
    
    if os.path.exists(CHECKPOINT_OUTPUT_DIR):
        files = os.listdir(CHECKPOINT_OUTPUT_DIR)
        for file in files:
            if file.startswith('model_withoutdynamic_checkpoint_epoch_') and file.endswith('.pt'):
                full_path = os.path.join(CHECKPOINT_OUTPUT_DIR, file)
                try:
                    epoch_num = int(file.split('_epoch_')[1].split('.pt')[0])
                    checkpoint_files.append((epoch_num, full_path))
                except:
                    continue
    
    return checkpoint_files

def load_latest_checkpoint(model):
    """Load checkpoint terbaru dari input atau working directory"""
    print("Checking for existing checkpoints...")
    
    # Cari checkpoint di kedua lokasi
    input_checkpoints = find_checkpoint_in_input()
    working_checkpoints = find_checkpoint_in_working()
    
    # Gabungkan dan cari yang terbaru
    all_checkpoints = input_checkpoints + working_checkpoints
    
    if not all_checkpoints:
        print("No checkpoint found. Starting from scratch.")
        return 0, [], []
    
    # Ambil checkpoint dengan epoch tertinggi
    latest_epoch, latest_path = max(all_checkpoints, key=lambda x: x[0])
    
    print(f"Found latest checkpoint: {latest_path} (epoch {latest_epoch})")
    try:
        from argparse import Namespace
        import torch.serialization
        torch.serialization.add_safe_globals([Namespace])
        checkpoint = torch.load(latest_path, map_location='cpu')
        model.load_state_dict(checkpoint['model_state_dict'])
        model.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        model.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        results_next_one = checkpoint.get('results_next_one', [])
        results_next_new = checkpoint.get('results_next_new', [])
        print(f"Resuming training from epoch {start_epoch}")
        return start_epoch, results_next_one, results_next_new
    except Exception as e:
        print(f"Error loading checkpoint: {e}")
        print("Starting training from scratch")
        return 0, [], []

def main():
    print('Resume Model Tanpa Preferensi Dinamis')
    print(f"Checkpoint input directory: {CHECKPOINT_INPUT_DIR}")
    print(f"Checkpoint output directory: {CHECKPOINT_OUTPUT_DIR}")

    opt.dataset = 'music4all'
    print(f'============= dataset: {opt.dataset} =============')

    n_music = 56047
    n_artist = 9348
    n_album = 21478
    n_usr = 8000

    print("--------------- Data Input ---------------")
    train_data_raw, test_data_raw, test_new_data_raw = data_input(opt)

    print(f"Total split train records: {len(train_data_raw[0])}")
    print(f"Total split test records (next-one): {len(test_data_raw[0])}")
    print(f"Total split test records (next-new): {len(test_new_data_raw[0])}")

    train_data = Data(train_data_raw, opt)
    test_data = Data(test_data_raw, opt)
    test_new_data = Data(test_new_data_raw, opt)

    model = trans_to_cuda(SessionGraph(opt, n_usr, n_album, n_artist, n_music))

    # Auto-load checkpoint jika ada
    start_epoch, results_next_one, results_next_new = load_latest_checkpoint(model)
    
    if start_epoch >= opt.epoch:
        print(f"Training already completed (epoch {start_epoch} >= {opt.epoch})")
        print("To continue training, increase --epoch parameter")
        return

    best_hit_next_one = None
    best_mrr_next_one = None
    best_hit_next_new = None
    best_mrr_next_new = None

    # Training loop dengan checkpoint otomatis
    for epoch in range(start_epoch, opt.epoch):
        print('-------------------------------------------------------')
        print(f'epoch: {epoch}')
        
        try:
            (hit_next_one, mrr_next_one), (hit_next_new, mrr_next_new) = train_test(model, train_data, test_data, test_new_data)
            sys.stdout.flush()

            best_hit_next_one = hit_next_one
            best_mrr_next_one = mrr_next_one
            best_hit_next_new = hit_next_new
            best_mrr_next_new = mrr_next_new

            result_one = {
                'epoch': epoch + 1,
                'hitrate10': float(best_hit_next_one[9]),
                'hitrate20': float(best_hit_next_one[19]),
                'mrr10': float(best_mrr_next_one[9]),
                'mrr20': float(best_mrr_next_one[19]),
            }
            result_new = {
                'epoch': epoch + 1,
                'hitrate10': float(best_hit_next_new[9]),
                'hitrate20': float(best_hit_next_new[19]),
                'mrr10': float(best_mrr_next_new[9]),
                'mrr20': float(best_mrr_next_new[19]),
            }

            results_next_one.append(result_one)
            results_next_new.append(result_new)

            # Auto-save checkpoint setiap N epoch
            if (epoch + 1) % SAVE_EVERY == 0:
                save_checkpoint(model, epoch, results_next_one, results_next_new)
                print(f"Auto-saved checkpoint at epoch {epoch + 1}")

        except Exception as e:
            print(f"Error during training at epoch {epoch}: {e}")
            # Simpan emergency checkpoint
            emergency_path = save_checkpoint(model, epoch - 1, results_next_one, results_next_new)
            print(f"Emergency checkpoint saved: {emergency_path}")
            raise e

    # Simpan final checkpoint jika training selesai
    if opt.epoch > 0:
        final_path = save_checkpoint(model, opt.epoch - 1, results_next_one, results_next_new)
        print(f"Final checkpoint saved: {final_path}")

    del model

    end_time = time.time()
    duration_hours = (end_time - start_time) / 3600
    print(f"\nTotal waktu yang dihabiskan: {duration_hours:.2f} jam")

    results_one_df = pd.DataFrame(results_next_one)
    results_new_df = pd.DataFrame(results_next_new)

    # Simpan hasil ke /kaggle/working dengan timestamp
    results_one_path = os.path.join(CHECKPOINT_OUTPUT_DIR, f'results_model_withoutdynamic_next_one.csv')
    results_new_path = os.path.join(CHECKPOINT_OUTPUT_DIR, f'results_model_withoutdynamic_next_new.csv')

    results_one_df.to_csv(results_one_path, index=False)
    results_new_df.to_csv(results_new_path, index=False)

    print(f"Saved results to:\n  {results_one_path}\n  {results_new_path}")

if __name__ == '__main__':
    main()

Resume Model Tanpa Preferensi Dinamis
Checkpoint input directory: /kaggle/input/checkpoint_model_withoutdynamic_epoch10/pytorch/default/1
Checkpoint output directory: /kaggle/working/
============= dataset: music4all =============
--------------- Data Input ---------------
Total split train records: 2471719
Total split test records (next-one): 603960
Total split test records (next-new): 272431
Checking for existing checkpoints...
Scanning checkpoint input directory: /kaggle/input/checkpoint_model_withoutdynamic_epoch10/pytorch/default/1
Files found: ['model_withoutdynamic_checkpoint_epoch_10.pt']
Found checkpoint: model_withoutdynamic_checkpoint_epoch_10.pt (epoch 10)
Found latest checkpoint: /kaggle/input/checkpoint_model_withoutdynamic_epoch10/pytorch/default/1/model_withoutdynamic_checkpoint_epoch_10.pt (epoch 10)
Resuming training from epoch 10
-------------------------------------------------------
epoch: 10
start training: 
[0/9656] Loss: 6.1089
[1932/9656] Loss: 6.1662
[3864/965